In [0]:
%run ./lib/00_common

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_STAGING}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_BRONZE}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}")

In [0]:
# Ahora crea el volumen (por defecto creará un "Managed Volume")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_STAGING}.{VOLUME}")

print("Volumen 'elt_volume' creado o verificado exitosamente.")

In [0]:

# Eliminar la tabla con el esquema viejo
spark.sql(f"DROP TABLE IF EXISTS {BRONZE_ORDERS_TABLE}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_ORDERS_TABLE} (
    order_id BIGINT,
    customer_id INT,
    order_date DATE,
    amount DOUBLE,
    status STRING,
    _ingestion_ts TIMESTAMP,
    _source_file STRING,
    _run_id STRING,
    _run_date STRING
) USING DELTA
""")

spark.sql(f"DROP TABLE IF EXISTS {BRONZE_CUSTOMERS_TABLE}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_CUSTOMERS_TABLE} (
    customer_id INT,
    customer_name STRING,
    country STRING,
    email STRING,
    updated_at TIMESTAMP,
    _ingestion_ts TIMESTAMP,
    _source_file STRING,
    _run_id STRING,
    _run_date STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {REJECT_ORDERS_TABLE} (
    order_id BIGINT,
    customer_id INT,
    order_date DATE,
    amount DOUBLE,
    status STRING,
    _ingestion_ts TIMESTAMP,
    _source_file STRING,
    _run_id STRING,
    _run_date STRING,
    reject_reason STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_ORDERS_TABLE} (
    order_id BIGINT,
    customer_id INT,
    order_date DATE,
    amount DOUBLE,
    status STRING,
    _ingestion_ts TIMESTAMP,
    _source_file STRING,
    _run_id STRING,
    _run_date STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_CUSTOMERS_TABLE} (
    customer_id INT,
    customer_name STRING,
    country STRING,
    email STRING,
    updated_at TIMESTAMP,
    _ingestion_ts TIMESTAMP,
    _source_file STRING,
    _run_id STRING,
    _run_date STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_DAILY_SALES_TABLE} (
    order_date DATE,
    total_orders BIGINT,
    total_revenue DOUBLE,
    avg_ticket DOUBLE
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_SALES_BY_COUNTRY_TABLE} (
    order_date DATE,
    country STRING,
    total_orders BIGINT,
    total_revenue DOUBLE
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {OPS_PIPELINE_EVENTS} (
    event_ts TIMESTAMP,
    level STRING,
    run_id STRING,
    stage STRING,
    dataset STRING,
    status STRING,
    message STRING,
    rows_ok BIGINT,
    rows_reject BIGINT,
    error_code STRING,
    extra STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {OPS_PIPELINE_ALERTS} (
    alert_ts TIMESTAMP,
    run_id STRING,
    alert_type STRING,
    severity STRING,
    message STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {OPS_DATASET_VERSIONS} (
    audit_ts TIMESTAMP,
    run_id STRING,
    table_name STRING,
    version BIGINT,
    operation STRING,
    operation_ts TIMESTAMP,
    num_output_rows STRING,
    user_name STRING
) USING DELTA
""")

print("Setup completado.")
print(f"Volume root: {VOLUME_ROOT}")